# Lesson 2 : Trace Agent

In actual development, you will likely need to perform detailed tracing and correct the context - such as, customizing instructions, setting-up appropriate tools, adding memories, etc.  
Agent Framework provides tracing and logging capabilities by using OpenTelemetry standard. You can configure to work with a wide variety of logging platforms - such as, Aspire Dashboard, Jaeger, Prometheus, etc. (You can also output trace and logs in your console.)

In this exercise, we'll trace a agent which is built in Lesson 1, and send logs to Azure Application Insights through Microsoft Foundry.

## 1. Create and connect to an Application Insights resource

In order for tracing, you should prepare an Application Insights resource in Microsoft Foundry.  
Before you start tracing, run the following steps.

1. Open [Azure Portal](https://portal.azure.com) and create a new Application Insights resource.
2. Open Foundry Portal (new portal), click "Operate" menu, and select "Admin" in left-side navigation.
3. Select your project, and connect to above Application Insights resource by clicking "Add connection".

## 2. Tracing your agent

Same as Lesson 1, we initialize a client object for the agent.  
In this exercise, however, please provide ```agent_name``` and ```use_latest_version``` as follows, because we connect to the existing agent in Microsoft Foundry.

> Note : Instead of setting ```agent_name``` and ```use_latest_version```, you can also set ```agent_name``` and ```agent_version``` when you use a specific version.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
    agent_name="BasicWeatherAgent",
    use_latest_version=True,
)

Connect to the existing agent as follows. (This code is mostly the same as Lesson 1.)

In [2]:
from typing import Annotated
from pydantic import Field
from random import randint
from agent_framework import Agent

# define local tools
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],  # "天気を取得する場所"
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]  # "晴れ", "曇り", "雨", "嵐"
    return f"The weather in {location} is {conditions[randint(0, 3)]}."  # "{location}の天気は{conditions[randint(0, 3)]}です。"

def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],  # "気温を取得する場所"
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."  # "{location}の気温は{randint(10, 30)}°Cです。"

# connect to the agent
agent = Agent(
    client=client,
    instructions="You are an agent about weather information.",  # "あなたは、気象情報に関する Agent です。"
    tools=[get_weather, get_temperature])

Run agent with trace enabled as follows.

For Foundry tracing with Application Insights, you can simply use built-in ```configure_azure_monitor()``` in client. (This method internally detects the connected Application Insights resource, invokes ```configure_azure_monitor()``` in Azure observability SDK, and invokes built-in ```enable_instrumentation()``` in Agent Framework SDK.)

> Note : For other tracing, you can use built-in ```configure_otel_providers()``` which reads related environments and configures. (You can also manually configure exporters, providers, and instrumentation for OpenTelemetry tracing.)  
> See [here](https://github.com/microsoft/agent-framework/tree/main/python/samples/getting_started/observability) for examples.

In [3]:
from IPython.display import Markdown, display
from agent_framework.observability import get_tracer
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

await client.configure_azure_monitor(
    enable_live_metrics=False,
    enable_sensitive_data=True,
)

with get_tracer().start_as_current_span("Weather Agent Test") as current_span:
    print(f"Trace ID: {format_trace_id(current_span.get_span_context().trace_id)}")

    result = await agent.run("Tell me the weather and temperature in Osaka.")  # "大阪の天気と気温を教えてください。"
    display(Markdown(result.text))

Trace ID: 35f2b2aa88729fbc965edd1b86a19ee1


Osaka: **Cloudy**, **11 °C**.

You can then view the collected trace in Application Insights or Azure Monitor. (For the first time, it may take a while...)  
For example, let's see and check logs calling OpenAI Responses API invoked from Agent Framework, as follows.

1. Go to Application Insights resource in [Azure Portal](https://portal.zure.com), and select your resource.
2. Expand "Investigate" in left-side navigator and click "Agents".
3. Click item in "Tool calls" or "Models" section.
4. Select a target trace session.
5. Expand call navigation in left-side, and select one of "LLM" items or "GenAI" items. (See below picture.)

![Trace view in Application Insights](./assets/appinsight_trace.png)

> Note : To view the input text and the generated output text, set ```enable_sensitive_data=True``` in ```configure_azure_monitor()```.